In [2]:
import sys; sys.path.append("..")
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from src.data import load_data
from src.config import RANDOM_STATE, CV_DIR
from src.experiment_log import ExperimentLogger

X_train, _, y_train, _, labels = load_data()

param_dist = {
    "learning_rate": [0.03, 0.05, 0.1, 0.2],
    "max_iter": [100, 200, 300],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [2, 5, 10, 20],
    "l2_regularization": [0.0, 0.1, 1.0],
}

cv3 = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)  # 3-fold cho nhanh
rs = RandomizedSearchCV(
    HistGradientBoostingClassifier(early_stopping=False, random_state=RANDOM_STATE),
    param_dist, n_iter=10, scoring="f1_macro", cv=cv3,
    random_state=RANDOM_STATE, n_jobs=1, verbose=2
)
rs.fit(X_train, y_train)

pd.DataFrame(rs.cv_results_).to_csv(CV_DIR / "tuning_hgb_random.csv", index=False)
print("best_params:", rs.best_params_, "| best_score (3-fold):", round(rs.best_score_, 4))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END l2_regularization=1.0, learning_rate=0.2, max_iter=300, max_leaf_nodes=31, min_samples_leaf=2; total time=  43.7s
[CV] END l2_regularization=1.0, learning_rate=0.2, max_iter=300, max_leaf_nodes=31, min_samples_leaf=2; total time=  44.9s
[CV] END l2_regularization=1.0, learning_rate=0.2, max_iter=300, max_leaf_nodes=31, min_samples_leaf=2; total time=  42.7s
[CV] END l2_regularization=0.0, learning_rate=0.1, max_iter=100, max_leaf_nodes=15, min_samples_leaf=20; total time=  10.3s
[CV] END l2_regularization=0.0, learning_rate=0.1, max_iter=100, max_leaf_nodes=15, min_samples_leaf=20; total time=   9.9s
[CV] END l2_regularization=0.0, learning_rate=0.1, max_iter=100, max_leaf_nodes=15, min_samples_leaf=20; total time=  10.1s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=100, max_leaf_nodes=15, min_samples_leaf=2; total time=  10.4s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=100, max_leaf

In [3]:
lg = ExperimentLogger("thai")
best = HistGradientBoostingClassifier(**rs.best_params_, early_stopping=False,
                                      random_state=RANDOM_STATE)
lg.run(best, X_train, y_train, "hgb_tuned", notes=f"RandomizedSearch 3-fold n_iter=10; cv={rs.best_score_:.4f}")

,accuracy,balanced_acc,macro_f1,weighted_f1,mcc,recall_c0,recall_c1,recall_c2,recall_c3,recall_c4,...,recall_c6,recall_c7,recall_c8,recall_c9,recall_c10,recall_c11,model,fold,fit_time,pred_time
0,0.998198,0.959175,0.967838,0.998194,0.995556,0.990984,0.964706,1.0,0.998489,0.833333,...,1.0,1.000000,0.987893,1.000000,0.991837,0.942857,hgb_tuned,0,14.429933,0.263517
1,0.997721,0.929716,0.933833,0.997705,0.994381,0.990984,0.988235,1.0,1.000000,0.500000,...,1.0,0.993789,0.985472,0.996885,0.986939,0.914286,hgb_tuned,1,11.757275,0.286766
2,0.997880,0.955255,0.951105,0.997888,0.994773,0.991803,0.988372,1.0,1.000000,0.800000,...,1.0,0.993789,0.985472,0.996885,0.985306,0.971429,hgb_tuned,2,11.513935,0.251088
3,0.998569,0.983367,0.978089,0.998575,0.996471,0.992623,1.000000,1.0,0.995475,0.833333,...,1.0,1.000000,0.985507,1.000000,0.993464,1.000000,hgb_tuned,3,14.213856,0.313040
4,0.998092,0.950813,0.969548,0.998082,0.995295,0.994262,0.941176,1.0,0.998492,0.833333,...,1.0,1.000000,0.992754,0.996894,0.988562,0.914286,hgb_tuned,4,12.919921,0.262151


In [ ]:
from sklearn.neural_network import MLPClassifier
from src.pipelines import scaled
# Khởi tạo Pipeline gồm bước chuẩn hóa dữ liệu và mô hình MLP
mlp_pipe = scaled(MLPClassifier(early_stopping=True, n_iter_no_change=10,
                                max_iter=200, random_state=RANDOM_STATE))
# Khai báo không gian siêu tham số cần tìm kiếm
param_mlp = {

    "clf__hidden_layer_sizes": [(64,), (128,), (128, 64)],
    "clf__alpha": [1e-5, 1e-4, 1e-3],
    "clf__learning_rate_init": [1e-3, 3e-3],
}
# Thiết lập tìm kiếm ngẫu nhiên
rs_m = RandomizedSearchCV(mlp_pipe, param_mlp, n_iter=6, scoring="f1_macro",
                          cv=cv3, random_state=RANDOM_STATE, n_jobs=1, verbose=2)

# Huấn luyện và tìm kiếm tổ hợp tham số tốt nhất trên tập huấn luyện
rs_m.fit(X_train, y_train)

# Lưu toàn bộ lịch sử và kết quả các lần thử nghiệm  
pd.DataFrame(rs_m.cv_results_).to_csv(CV_DIR / "tuning_mlp_random.csv", index=False)
print(rs_m.best_params_, round(rs_m.best_score_, 4))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END clf__alpha=1e-05, clf__hidden_layer_sizes=(64,), clf__learning_rate_init=0.001; total time=  14.6s
[CV] END clf__alpha=1e-05, clf__hidden_layer_sizes=(64,), clf__learning_rate_init=0.001; total time=  10.7s
[CV] END clf__alpha=1e-05, clf__hidden_layer_sizes=(64,), clf__learning_rate_init=0.001; total time=  11.7s
[CV] END clf__alpha=1e-05, clf__hidden_layer_sizes=(64,), clf__learning_rate_init=0.003; total time=   8.1s
[CV] END clf__alpha=1e-05, clf__hidden_layer_sizes=(64,), clf__learning_rate_init=0.003; total time=  10.9s
[CV] END clf__alpha=1e-05, clf__hidden_layer_sizes=(64,), clf__learning_rate_init=0.003; total time=  12.5s
[CV] END clf__alpha=0.0001, clf__hidden_layer_sizes=(128,), clf__learning_rate_init=0.001; total time=  12.6s
[CV] END clf__alpha=0.0001, clf__hidden_layer_sizes=(128,), clf__learning_rate_init=0.001; total time=  19.1s
[CV] END clf__alpha=0.0001, clf__hidden_layer_sizes=(128,), clf__learnin